# 04 — Synthetic Stress

**Benchmark:** Benchmark 3 — Synthetic Stress Benchmark  
**Scope:** Synthetic stress scenarios: `100M` real-like scale, `10M_skewed`, and `10M_highuid`.  
**Frameworks:** Pandas, Polars eager, Polars lazy, Dask.

Notebook này không chạy benchmark workload. Notebook chỉ đọc kết quả đã có trong `results/raw/` và phân tích robustness của framework dưới các điều kiện synthetic khó: row count rất lớn, key skew mạnh, và ID cardinality cao.

Điểm quan trọng của notebook này là **phân biệt dữ liệu không chạy vì giới hạn tài nguyên** với dữ liệu bị thiếu. Với `100M`, Pandas chắc chắn không chạy nổi trên máy hiện tại; Polars eager đã được thử và làm máy shutdown; join ở `100M` cũng quá lớn và làm máy shutdown. Vì vậy phần `100M` chỉ phân tích `filter` và `groupby` cho Polars lazy và Dask.

## 1. Research Questions

Notebook này trả lời sáu câu hỏi chính:

1. Ở synthetic `100M` real-like, Polars lazy và Dask xử lý `filter`/`groupby` tốt đến mức nào?
2. Kết quả `100M` này nằm ở đâu nếu đặt cạnh mốc real data `1M`, `10M`, `50M` của cùng framework?
3. Với `10M_skewed`, framework nào nhạy nhất với key skew?
4. Với `10M_highuid`, framework nào bị ảnh hưởng nhiều nhất bởi cardinality cao?
5. Stress synthetic làm runtime và memory thay đổi bao nhiêu so với mốc real `10M` trong cùng framework/workload?
6. Các giới hạn tài nguyên quan sát được nói gì về phạm vi dùng thực tế của Pandas, Polars eager, Polars lazy và Dask?

## 2. Setup and Data Loading

Code dùng key và tên biến tiếng Anh để khớp với CSV benchmark. Nội dung markdown dùng tiếng Việt để giải thích kết quả.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() in {"benchmark", "benchmarks", "analysis", "report", "data_prep"}:
    PROJECT_ROOT = PROJECT_ROOT.parent.parent.resolve()
elif PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

RESULTS_DIR = PROJECT_ROOT / "results" / "raw"
FIGURE_DIR = PROJECT_ROOT / "results" / "figures" / "notebook_04_synthetic_stress"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

STRESS_FILES = [
    RESULTS_DIR / "dask_100m_results.csv",
    RESULTS_DIR / "polars_100m_results.csv",
    RESULTS_DIR / "dask_highuid_skewed_results.csv",
    RESULTS_DIR / "pandas_highuid_skewed_results.csv",
    RESULTS_DIR / "polars_highuid_skewed_results.csv",
    RESULTS_DIR / "polars_eager_highuid_skewed_results.csv",
]

REAL_BASELINE_FILES = [
    RESULTS_DIR / "dask_real_results.csv",
    RESULTS_DIR / "pandas_real_results.csv",
    RESULTS_DIR / "polars_real_results.csv",
    RESULTS_DIR / "polars_eager_real_results.csv",
]

FRAMEWORK_ORDER = ["pandas", "polars_eager", "polars_lazy", "dask"]
WORKLOAD_ORDER = ["filter", "groupby", "join", "pipeline"]
ROW_ORDER = ["1M", "10M", "50M", "100M"]
STRESS_ORDER = ["10M_skewed", "10M_highuid"]
FRAMEWORK_LABELS = {
    "pandas": "Pandas",
    "polars_eager": "Polars eager",
    "polars_lazy": "Polars lazy",
    "dask": "Dask",
}

In [ ]:
available_files = [path for path in STRESS_FILES + REAL_BASELINE_FILES if path.exists()]
missing_files = [path.name for path in STRESS_FILES + REAL_BASELINE_FILES if not path.exists()]

if missing_files:
    print("Missing files:", missing_files)

if not available_files:
    raise FileNotFoundError("No benchmark CSV files were found in results/raw/.")

frames = []
for path in available_files:
    frame = pd.read_csv(path)
    frame["source_file"] = path.name
    if "100m" in path.name.lower() or "highuid_skewed" in path.name.lower():
        frame["benchmark_family"] = "synthetic_stress"
    elif "real" in path.name.lower():
        frame["benchmark_family"] = "real_baseline"
    else:
        frame["benchmark_family"] = "other"
    frames.append(frame)

df_raw = pd.concat(frames, ignore_index=True)

required_columns = {
    "timestamp", "framework", "workload", "dataset_size", "n_rows",
    "run_index", "time_s", "peak_memory_mb", "throughput_rows_per_s", "status", "notes"
}
missing_columns = sorted(required_columns - set(df_raw.columns))
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

for col in ["time_s", "peak_memory_mb", "throughput_rows_per_s", "n_rows"]:
    df_raw[col] = pd.to_numeric(df_raw[col], errors="coerce")

df_raw["framework"] = pd.Categorical(df_raw["framework"], FRAMEWORK_ORDER, ordered=True)
df_raw["workload"] = pd.Categorical(df_raw["workload"], WORKLOAD_ORDER, ordered=True)
df_raw["dataset_size"] = df_raw["dataset_size"].astype(str)

def scenario_label(dataset_size):
    if dataset_size == "100M":
        return "100M_real_like"
    if dataset_size == "10M_skewed":
        return "heavy_skew_10M"
    if dataset_size == "10M_highuid":
        return "high_unique_id_10M"
    return "real_baseline"

df_raw["stress_scenario"] = df_raw["dataset_size"].map(scenario_label)

display(df_raw.head())
print(f"Loaded rows: {len(df_raw):,}")
print(f"Sources: {', '.join(sorted(df_raw['source_file'].unique()))}")

## 3. Data Completeness Check

Phân tích runtime/memory chính chỉ dùng `status == "ok"`. Những trường hợp không có kết quả ở `100M` được ghi riêng vì đây là giới hạn thực nghiệm đã quan sát được, không phải lỗi load dữ liệu.

In [ ]:
df_ok = df_raw[df_raw["status"] == "ok"].copy()
df_fail = df_raw[df_raw["status"] != "ok"].copy()

run_count = (
    df_raw
    .groupby(["benchmark_family", "framework", "dataset_size", "workload", "status"], observed=True)
    .size()
    .rename("n_runs")
    .reset_index()
    .sort_values(["benchmark_family", "dataset_size", "framework", "workload"])
)

display(run_count)
print(f"Successful rows: {len(df_ok):,}")
print(f"Failed rows    : {len(df_fail):,}")

In [ ]:
stress_completion = (
    df_ok[df_ok["benchmark_family"] == "synthetic_stress"]
    .groupby(["dataset_size", "framework", "workload"], observed=True)["run_index"]
    .count()
    .rename("successful_runs")
    .reset_index()
    .pivot_table(
        index=["dataset_size", "framework"],
        columns="workload",
        values="successful_runs",
        fill_value=0,
        observed=True,
    )
)

display(stress_completion)

### Methodological boundary

`100M` không được đọc như một ma trận benchmark đầy đủ. Đây là một stress-scale probe có chủ đích:

- Pandas `100M`: không chạy vì vượt giới hạn thực tế của máy.
- Polars eager `100M`: đã thử và làm máy shutdown.
- Join `100M`: output/intermediate quá lớn, đã gây shutdown, nên không đưa vào benchmark chính.
- Vì vậy `100M` chỉ hợp lệ cho so sánh `filter` và `groupby` giữa Polars lazy và Dask.

Các stress case `10M_skewed` và `10M_highuid` thì có đủ Pandas, Polars eager, Polars lazy và Dask cho `filter`, `groupby`, `join`, `pipeline`.

In [ ]:
omitted_100m = pd.DataFrame([
    {
        "scenario": "100M_real_like",
        "framework": "pandas",
        "workload": "filter/groupby/join/pipeline",
        "status": "not_run",
        "reason": "Pandas 100M is outside the practical memory limit on this machine.",
    },
    {
        "scenario": "100M_real_like",
        "framework": "polars_eager",
        "workload": "filter/groupby/join/pipeline",
        "status": "aborted",
        "reason": "Polars eager 100M was attempted and caused the machine to shut down.",
    },
    {
        "scenario": "100M_real_like",
        "framework": "polars_lazy/dask",
        "workload": "join",
        "status": "aborted",
        "reason": "100M join materialization/intermediate size was too large and caused shutdown.",
    },
    {
        "scenario": "100M_real_like",
        "framework": "polars_lazy/dask",
        "workload": "pipeline",
        "status": "not_in_scope",
        "reason": "Pipeline is excluded from 100M stress interpretation because join-like intermediate pressure is too risky.",
    },
])

display(omitted_100m)

## 4. Summary Table

Bảng `summary` là nguồn chính cho các phân tích tiếp theo. Mỗi tổ hợp dùng mean, standard deviation và số run thành công.

In [ ]:
summary = (
    df_ok
    .groupby(["benchmark_family", "stress_scenario", "dataset_size", "framework", "workload"], observed=True)
    .agg(
        n_rows=("n_rows", "median"),
        n_runs=("run_index", "count"),
        time_mean_s=("time_s", "mean"),
        time_std_s=("time_s", "std"),
        memory_mean_mb=("peak_memory_mb", "mean"),
        memory_std_mb=("peak_memory_mb", "std"),
        throughput_mean_rows_s=("throughput_rows_per_s", "mean"),
        throughput_std_rows_s=("throughput_rows_per_s", "std"),
    )
    .reset_index()
)

summary["time_cv"] = summary["time_std_s"] / summary["time_mean_s"]
summary["memory_gb"] = summary["memory_mean_mb"] / 1024
summary["rows_per_second_m"] = summary["throughput_mean_rows_s"] / 1_000_000
summary["seconds_per_million_rows"] = summary["time_mean_s"] / (summary["n_rows"] / 1_000_000)

summary_display = summary.copy()
for col in ["time_mean_s", "time_std_s", "memory_mean_mb", "memory_std_mb", "time_cv", "memory_gb", "rows_per_second_m", "seconds_per_million_rows"]:
    summary_display[col] = summary_display[col].round(3)

display(summary_display.sort_values(["benchmark_family", "dataset_size", "workload", "framework"]))

## 5. `100M` Real-like Stress Scale

Phần này chỉ so sánh Polars lazy và Dask trên `filter` và `groupby`. Đây là câu hỏi về khả năng đi qua scale rất lớn trong điều kiện còn an toàn cho máy, không phải benchmark đầy đủ mọi workload.

In [ ]:
stress_100m = summary[
    (summary["benchmark_family"] == "synthetic_stress")
    & (summary["dataset_size"] == "100M")
    & (summary["workload"].isin(["filter", "groupby"]))
].copy()

stress_100m_display = stress_100m[[
    "framework", "workload", "n_runs", "time_mean_s", "time_std_s",
    "memory_gb", "rows_per_second_m", "seconds_per_million_rows"
]].copy()
for col in ["time_mean_s", "time_std_s", "memory_gb", "rows_per_second_m", "seconds_per_million_rows"]:
    stress_100m_display[col] = stress_100m_display[col].round(3)

display(stress_100m_display.sort_values(["workload", "framework"]))

In [ ]:
def grouped_bar(data, x_col, y_col, hue_col, title, ylabel, filename, order=None, hue_order=None):
    plot_data = data.copy()
    if order is None:
        order = list(plot_data[x_col].dropna().unique())
    if hue_order is None:
        hue_order = list(plot_data[hue_col].dropna().unique())

    x = np.arange(len(order))
    width = 0.8 / max(len(hue_order), 1)

    fig, ax = plt.subplots(figsize=(9, 5))
    for i, hue in enumerate(hue_order):
        subset = plot_data[plot_data[hue_col] == hue]
        values = []
        errors = []
        for item in order:
            row = subset[subset[x_col] == item]
            if row.empty:
                values.append(np.nan)
                errors.append(0)
            else:
                values.append(row[y_col].iloc[0])
                err_col = y_col.replace("mean", "std")
                errors.append(row[err_col].iloc[0] if err_col in row.columns else 0)
        ax.bar(x + (i - (len(hue_order) - 1) / 2) * width, values, width, label=FRAMEWORK_LABELS.get(str(hue), str(hue)), yerr=errors, capsize=3)

    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xticks(x)
    ax.set_xticklabels(order)
    ax.legend(frameon=False)
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    out = FIGURE_DIR / filename
    fig.savefig(out, dpi=160, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out}")

grouped_bar(
    stress_100m,
    x_col="workload",
    y_col="time_mean_s",
    hue_col="framework",
    title="100M synthetic stress: runtime",
    ylabel="Mean runtime (seconds)",
    filename="100m_runtime.png",
    order=["filter", "groupby"],
    hue_order=["polars_lazy", "dask"],
)

grouped_bar(
    stress_100m,
    x_col="workload",
    y_col="memory_mean_mb",
    hue_col="framework",
    title="100M synthetic stress: peak memory",
    ylabel="Mean peak memory (MB)",
    filename="100m_memory.png",
    order=["filter", "groupby"],
    hue_order=["polars_lazy", "dask"],
)

### What this means

Với `100M`, Polars lazy hoàn thành cả `filter` và `groupby` nhanh hơn Dask trong dữ liệu hiện có. Chênh lệch lớn nhất thường nằm ở `filter`, nơi chi phí scan, predicate, materialization và scheduler overhead đều cùng xuất hiện.

Memory lại không cùng một chiều cho mọi workload: `filter` cần materialize nhiều dòng hơn nên peak memory cao hơn `groupby`; `groupby` có output nhỏ hơn nhưng vẫn chịu chi phí hash aggregation. Đây là lý do notebook này không suy diễn kết quả `100M` sang join hoặc pipeline.

## 6. `100M` Compared with Real `1M`/`10M`/`50M`

Phần này đặt `100M` synthetic cạnh mốc real data của cùng framework. Đây không phải so sánh “real vs synthetic” trực tiếp; mục đích là tạo mốc trực quan để thấy `100M` nằm ngoài dải scale real hiện có như thế nào.

In [ ]:
real_scale = summary[
    (summary["benchmark_family"] == "real_baseline")
    & (summary["framework"].isin(["polars_lazy", "dask"]))
    & (summary["workload"].isin(["filter", "groupby"]))
    & (summary["dataset_size"].isin(["1M", "10M", "50M"]))
].copy()

scale_compare = pd.concat([real_scale, stress_100m], ignore_index=True)
scale_compare["row_label"] = pd.Categorical(scale_compare["dataset_size"], ROW_ORDER, ordered=True)
scale_compare = scale_compare.sort_values(["workload", "framework", "row_label"])

scale_display = scale_compare[[
    "benchmark_family", "dataset_size", "framework", "workload", "n_rows",
    "time_mean_s", "memory_gb", "rows_per_second_m", "seconds_per_million_rows"
]].copy()
for col in ["time_mean_s", "memory_gb", "rows_per_second_m", "seconds_per_million_rows"]:
    scale_display[col] = scale_display[col].round(3)

display(scale_display)

In [ ]:
for workload in ["filter", "groupby"]:
    fig, ax = plt.subplots(figsize=(8, 5))
    subset = scale_compare[scale_compare["workload"] == workload]
    for framework in ["polars_lazy", "dask"]:
        fw = subset[subset["framework"] == framework].sort_values("n_rows")
        ax.plot(
            fw["n_rows"],
            fw["time_mean_s"],
            marker="o",
            linewidth=2,
            label=FRAMEWORK_LABELS.get(framework, framework),
        )
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_title(f"Real row scaling plus 100M synthetic marker: {workload}")
    ax.set_xlabel("Rows")
    ax.set_ylabel("Mean runtime (seconds, log scale)")
    ax.grid(True, which="both", alpha=0.25)
    ax.legend(frameon=False)
    fig.tight_layout()
    out = FIGURE_DIR / f"real_scale_plus_100m_{workload}.png"
    fig.savefig(out, dpi=160, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out}")

In [ ]:
# Compare 100M synthetic against 50M real in the same framework/workload.
real_50m = real_scale[real_scale["dataset_size"] == "50M"][["framework", "workload", "time_mean_s", "memory_mean_mb"]]
real_50m = real_50m.rename(columns={"time_mean_s": "real_50m_time_s", "memory_mean_mb": "real_50m_memory_mb"})

compare_100m_vs_50m = stress_100m.merge(real_50m, on=["framework", "workload"], how="left")
compare_100m_vs_50m["time_ratio_vs_real_50m"] = compare_100m_vs_50m["time_mean_s"] / compare_100m_vs_50m["real_50m_time_s"]
compare_100m_vs_50m["memory_ratio_vs_real_50m"] = compare_100m_vs_50m["memory_mean_mb"] / compare_100m_vs_50m["real_50m_memory_mb"]

ratio_display = compare_100m_vs_50m[[
    "framework", "workload", "time_mean_s", "real_50m_time_s", "time_ratio_vs_real_50m",
    "memory_gb", "real_50m_memory_mb", "memory_ratio_vs_real_50m"
]].copy()
ratio_display["real_50m_memory_gb"] = ratio_display["real_50m_memory_mb"] / 1024
ratio_display = ratio_display.drop(columns=["real_50m_memory_mb"])
for col in ["time_mean_s", "real_50m_time_s", "time_ratio_vs_real_50m", "memory_gb", "real_50m_memory_gb", "memory_ratio_vs_real_50m"]:
    ratio_display[col] = ratio_display[col].round(3)

display(ratio_display.sort_values(["workload", "framework"]))

### What this means

Mốc `100M` giúp thấy scale ceiling rõ hơn: kết quả vẫn hoàn thành với Polars lazy và Dask cho hai workload scan/aggregation, nhưng đây là vùng mà lựa chọn workload trở nên quyết định. `filter` giữ lại nhiều dữ liệu hơn nên tạo áp lực materialization; `groupby` thường trả output nhỏ hơn nên có thể chạy nhanh hơn dù input lớn.

So với mốc real `50M`, tỷ lệ runtime không nên đọc như scaling law chính xác vì dữ liệu khác nguồn và physical footprint có thể khác. Nó chỉ là reference marker để trả lời: “nếu real benchmark hiện dừng ở `50M`, thì `100M` synthetic đang nằm ở vùng runtime/memory nào?”.

## 7. `10M_skewed` and `10M_highuid` Stress Summary

Hai stress case `10M` có thể so sánh trong cùng framework và cùng workload với baseline real `10M`. Cách đọc chính ở đây là slowdown hoặc speedup tương đối, không phải ranking tuyệt đối duy nhất.

In [ ]:
stress_10m = summary[
    (summary["benchmark_family"] == "synthetic_stress")
    & (summary["dataset_size"].isin(STRESS_ORDER))
].copy()

real_10m = summary[
    (summary["benchmark_family"] == "real_baseline")
    & (summary["dataset_size"] == "10M")
].copy()

baseline_10m = real_10m[["framework", "workload", "time_mean_s", "memory_mean_mb"]].rename(
    columns={"time_mean_s": "real_10m_time_s", "memory_mean_mb": "real_10m_memory_mb"}
)

stress_vs_real_10m = stress_10m.merge(baseline_10m, on=["framework", "workload"], how="left")
stress_vs_real_10m["time_ratio_vs_real_10m"] = stress_vs_real_10m["time_mean_s"] / stress_vs_real_10m["real_10m_time_s"]
stress_vs_real_10m["memory_ratio_vs_real_10m"] = stress_vs_real_10m["memory_mean_mb"] / stress_vs_real_10m["real_10m_memory_mb"]
stress_vs_real_10m["stress_label"] = pd.Categorical(stress_vs_real_10m["dataset_size"], STRESS_ORDER, ordered=True)

stress_ratio_display = stress_vs_real_10m[[
    "dataset_size", "framework", "workload", "n_runs",
    "time_mean_s", "real_10m_time_s", "time_ratio_vs_real_10m",
    "memory_gb", "memory_ratio_vs_real_10m"
]].copy()
for col in ["time_mean_s", "real_10m_time_s", "time_ratio_vs_real_10m", "memory_gb", "memory_ratio_vs_real_10m"]:
    stress_ratio_display[col] = stress_ratio_display[col].round(3)

display(stress_ratio_display.sort_values(["dataset_size", "workload", "framework"]))

In [ ]:
for metric, ylabel, filename_prefix in [
    ("time_mean_s", "Mean runtime (seconds)", "stress10m_runtime"),
    ("memory_mean_mb", "Mean peak memory (MB)", "stress10m_memory"),
]:
    for workload in WORKLOAD_ORDER:
        subset = stress_10m[stress_10m["workload"] == workload]
        if subset.empty:
            continue
        grouped_bar(
            subset,
            x_col="dataset_size",
            y_col=metric,
            hue_col="framework",
            title=f"10M stress: {workload} — {ylabel}",
            ylabel=ylabel,
            filename=f"{filename_prefix}_{workload}.png",
            order=STRESS_ORDER,
            hue_order=FRAMEWORK_ORDER,
        )

### What this means

`10M_skewed` và `10M_highuid` không chỉ kiểm tra tốc độ; chúng kiểm tra độ nhạy với phân phối dữ liệu. Nếu một framework nhanh trên real `10M` nhưng slowdown mạnh khi key bị skew hoặc cardinality tăng, điều đó cho thấy ranking trên dữ liệu bình thường không đủ để dự đoán stress behavior.

Vì cả hai stress case đều có cùng row count `10M`, so sánh trong từng framework/workload là hợp lệ hơn so với trộn trực tiếp với `100M`.

## 8. Stress Slowdown Relative to Real `10M`

Biểu đồ sau chuẩn hóa runtime theo mốc real `10M` của chính framework đó. Giá trị `1.0` nghĩa là stress case chạy ngang với real `10M`; lớn hơn `1.0` là chậm hơn; nhỏ hơn `1.0` là nhanh hơn.

In [ ]:
for workload in WORKLOAD_ORDER:
    subset = stress_vs_real_10m[stress_vs_real_10m["workload"] == workload].copy()
    if subset.empty:
        continue
    grouped_bar(
        subset,
        x_col="dataset_size",
        y_col="time_ratio_vs_real_10m",
        hue_col="framework",
        title=f"Runtime ratio vs real 10M: {workload}",
        ylabel="Stress runtime / real 10M runtime",
        filename=f"stress_ratio_vs_real_10m_{workload}.png",
        order=STRESS_ORDER,
        hue_order=FRAMEWORK_ORDER,
    )

In [ ]:
# Compact ranking table: lower runtime ratio means less slowdown relative to the same framework's real 10M baseline.
ratio_rank = stress_vs_real_10m.copy()
ratio_rank["rank_within_stress_workload"] = ratio_rank.groupby(
    ["dataset_size", "workload"], observed=True
)["time_ratio_vs_real_10m"].rank(method="min")

ranking_display = ratio_rank[[
    "dataset_size", "workload", "framework", "time_ratio_vs_real_10m",
    "memory_ratio_vs_real_10m", "rank_within_stress_workload"
]].copy()
for col in ["time_ratio_vs_real_10m", "memory_ratio_vs_real_10m"]:
    ranking_display[col] = ranking_display[col].round(3)

display(ranking_display.sort_values(["dataset_size", "workload", "rank_within_stress_workload"]))

### Interpretation rule

Ranking theo `time_ratio_vs_real_10m` trả lời câu hỏi “framework này bị stress làm chậm đi bao nhiêu so với chính nó?”. Ranking theo `time_mean_s` trả lời câu hỏi khác: “framework nào nhanh nhất tuyệt đối?”. Notebook này cần cả hai vì stress benchmark quan tâm đến robustness, không chỉ tốc độ thô.

## 9. Polars Lazy vs Eager Under Stress

Polars có hai execution mode trong các stress case `10M`: eager và lazy. Phần này tách riêng để xem lazy optimization có giúp trong skew/high-cardinality không.

In [ ]:
polars_modes = stress_10m[stress_10m["framework"].isin(["polars_eager", "polars_lazy"])].copy()
polars_pivot = polars_modes.pivot_table(
    index=["dataset_size", "workload"],
    columns="framework",
    values=["time_mean_s", "memory_mean_mb"],
    observed=True,
)
polars_pivot.columns = ["_".join(col).strip() for col in polars_pivot.columns.to_flat_index()]
polars_pivot = polars_pivot.reset_index()
polars_pivot["lazy_speedup_vs_eager"] = polars_pivot["time_mean_s_polars_eager"] / polars_pivot["time_mean_s_polars_lazy"]
polars_pivot["lazy_memory_ratio_vs_eager"] = polars_pivot["memory_mean_mb_polars_lazy"] / polars_pivot["memory_mean_mb_polars_eager"]

polars_display = polars_pivot.copy()
for col in polars_display.columns:
    if col not in ["dataset_size", "workload"]:
        polars_display[col] = polars_display[col].round(3)

display(polars_display.sort_values(["dataset_size", "workload"]))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plot_data = polars_pivot.copy()
plot_data["case"] = plot_data["dataset_size"].astype(str) + " / " + plot_data["workload"].astype(str)
ax.bar(plot_data["case"], plot_data["lazy_speedup_vs_eager"])
ax.axhline(1.0, color="black", linewidth=1, linestyle="--")
ax.set_title("Polars lazy speedup vs eager under 10M stress")
ax.set_ylabel("Eager runtime / lazy runtime")
ax.set_xticklabels(plot_data["case"], rotation=45, ha="right")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
out = FIGURE_DIR / "polars_lazy_speedup_vs_eager_stress10m.png"
fig.savefig(out, dpi=160, bbox_inches="tight")
plt.show()
print(f"Saved: {out}")

### What this means

Polars lazy không nên được mô tả đơn giản là “luôn nhanh hơn”. Trong các stress case, lợi ích phụ thuộc workload: lazy có thể thắng khi optimizer giảm được work hoặc tận dụng execution plan tốt hơn, nhưng eager có thể rất cạnh tranh ở thao tác đơn giản hoặc khi overhead lazy không bù được.

Ở `100M`, kết luận lại khác: Polars eager không nằm trong phạm vi an toàn của máy, nên practical recommendation cho scale cực lớn là dùng Polars lazy/streaming-style execution hoặc Dask, không dùng eager materialization.

## 10. Failure and Scope Summary

Bảng dưới đây gom lại scope thực nghiệm để tránh diễn giải quá mức.

In [ ]:
observed_scope = pd.DataFrame([
    {"scenario": "100M_real_like", "valid_frameworks": "polars_lazy, dask", "valid_workloads": "filter, groupby", "interpretation": "Large-row stress probe only."},
    {"scenario": "10M_skewed", "valid_frameworks": "pandas, polars_eager, polars_lazy, dask", "valid_workloads": "filter, groupby, join, pipeline", "interpretation": "Compare skew sensitivity within each framework and workload."},
    {"scenario": "10M_highuid", "valid_frameworks": "pandas, polars_eager, polars_lazy, dask", "valid_workloads": "filter, groupby, join, pipeline", "interpretation": "Compare cardinality pressure within each framework and workload."},
])

display(observed_scope)
display(omitted_100m)

if len(df_fail):
    display(df_fail)
else:
    print("No explicit failed rows are recorded in the CSV files. Resource-limit cases are documented as omitted/aborted runs above.")

## 11. Key Takeaways

1. `100M` synthetic real-like chỉ nên được đọc trong phạm vi `filter` và `groupby`. Pandas, Polars eager và join/pipeline `100M` bị loại khỏi phân tích chính vì giới hạn tài nguyên thực tế, không phải vì notebook thiếu dữ liệu.
2. Polars lazy và Dask đều hoàn thành `100M` cho hai workload hợp lệ; Polars lazy nhanh hơn trong kết quả hiện có, còn Dask vẫn là mốc quan trọng cho partitioned execution.
3. `filter` ở scale lớn có áp lực materialization cao hơn `groupby`, nên memory/runtimes của `filter` không thể dùng để suy ra join safety.
4. `10M_skewed` và `10M_highuid` nên so với real `10M` trong cùng framework/workload để đo stress sensitivity.
5. Lazy vs eager trong Polars phụ thuộc workload ở `10M`, nhưng ở `100M` eager không còn là lựa chọn thực tế trên máy hiện tại.
6. Synthetic stress benchmark bổ sung cho real row scaling: nó chỉ ra failure modes và robustness limits mà real `1M`/`10M`/`50M` không bộc lộ đầy đủ.